In [86]:
import pandas as pd
import os

__version__ = "0.2.0"
length_unit= "cm"
time_unit= "days"
mass_units= "mmol"
print_screen= False

'''
class Model:
    def __init__(self,  name="model"):
        self.name = name

        self.basic_info = {
            "iVer": "4",
            "Hed": f"Created with Pydrus version {__version__}",
            "LUnit": length_unit,
            "TUnit": time_unit,
            "MUnit": mass_units,
            "lWat": False,
            "lChem": False,
            "lTemp": False,
            "lSink": False,
            "lRoot": False,
            "lShort": True,
            "lWDep": False,
            "lScreen": print_screen,
            "AtmInf": False,
            "lEquil": True,
            "lInverse": False,
            "lSnow": False,
            "lHP1": False,
            "lMeteo": False,
            "lVapor": False,
            "lActRSU": False,
            "lFlux": False,
            "lIrrig": False,
            "CosAlfa": 1,
        }
'''
        
# Create Header string
string = "*** BLOCK {:{}{}{}}\n"

info = pd.DataFrame()
info['Hed'] = f"Created with Pydrus version {__version__}"
info['description'] = "this is a test"
info['iVer'] = "4"

# Write block A: BASIC INFORMATION
lines = [f"Pcp_File_Version={info['iVer']}\n"
         f"{string.format('A: BASIC INFORMATION ', '*', '<', 72)}"
         f"{info['Hed']}\n{info['description']}\n"
         #f"LUnit TUnit MUnit\n{self.basic_info['LUnit']}\n"
         #f"{self.basic_info['TUnit']}\n{self.basic_info['MUnit']}\n"
        ]


days= list(range(20))
depths =list(range(20,40))


for i in list(range(20)):
    lines.append(str(days[i]))
    lines.append(" ")
    lines.append(str(depths[i]))
    lines.append("\n")

fname = os.path.join(os.pardir, '3_Results', 'TestingInputFiles', 'SELECTOR.IN')
with open(fname, "w") as file:
    file.writelines(lines)


In [ ]:
import pandas as pd
import os

# Function for just generating root growth part of text file (just to figure out how)

def selector_test(days, depths, irootin):
    string = "*** BLOCK {:{}{}{}}\n"
    lines = [f"Pcp_File_Version=\n"
             f"{string.format('A: BASIC INFORMATION ', '*', '<', 72)}"
         #f"LUnit TUnit MUnit\n{self.basic_info['LUnit']}\n"
         #f"{self.basic_info['TUnit']}\n{self.basic_info['MUnit']}\n"
            ]
    
    totaldays = len(days)
    
    lines.append("iRootDepthEntry\n")
    lines.append(f"{str(irootin)}\n")
    lines.append("nGrowth\n")
    lines.append(f"{str(totaldays)}\n")

    width = len(str(max(days)))

    lines.append(str("Time RootDepth\n"))

    for i in list(range(len(days))):
        lines.append(str(days[i]+1).ljust(width))
        lines.append(" ")
        lines.append(str(depths[i]))
        lines.append("\n")
        
    fname = os.path.join(os.pardir, '3_Results', 'TestingInputFiles', 'SELECTOR.IN')
    
    with open(fname, "w") as file:
        file.writelines(lines)

days= list(range(40))
depths =list(range(10,50))
irootin = 1

selector_test(days= days,
              depths=depths,
              irootin= irootin)

In [36]:
string = "*** BLOCK {:{}{}{}}\n"
lines = [f"Pcp_File_Version=\n"
         f"{string.format('A: BASIC INFORMATION ', '*', '<', 72)}"
         #f"LUnit TUnit MUnit\n{self.basic_info['LUnit']}\n"
         #f"{self.basic_info['TUnit']}\n{self.basic_info['MUnit']}\n"
        ]
root_growth = {
                "iRootIn": irootin,
                "nGrowth": max(days),
                "tGrowth": days,
                "RootDepth": depths
            }

d= root_growth.copy() #creating copy of selector input info
d.pop("iRootIn") #removing the irootin value
d['tGrowth'][2]

2

In [ ]:
#PHYdus official version of selector function

def write_selector_modified(self, fname="SELECTOR.IN"):
        """
        Write the SELECTOR.IN file.

        Parameters
        ----------
        fname: str, optional
            String with the filename. Written to the workspace folder ('ws').

        """
        self._set_bc_settings()

        # Create Header string
        string = "*** BLOCK {:{}{}{}}\n"

        # Write block A: BASIC INFORMATION
        lines = [
            f"Pcp_File_Version={self.basic_info['iVer']}\n"
            f"{string.format('A: BASIC INFORMATION ', '*', '<', 72)}"
            f"{self.basic_info['Hed']}\n{self.description}\n"
            f"LUnit TUnit MUnit\n{self.basic_info['LUnit']}\n"
            f"{self.basic_info['TUnit']}\n{self.basic_info['MUnit']}\n"
        ]

        vars_list = [["lWat", "lChem", "lTemp", "lSink", "lRoot", "lShort",
                      "lWDep", "lScreen", "AtmInf", "lEquil", "lInverse",
                      "\n"],
                     ["lSnow", "lHP1", "lMeteo", "lVapor", "lActRSU", "lFlux",
                      "lIrrig", "\n"]]

        for variables in vars_list:
            lines.append("  ".join(variables))
            lines.append("  ".join("t" if self.basic_info[var] else "f" for
                                   var in variables[:-1]))
            lines.append("\n")

        lines.append(f"NMat NLay CosAlfa \n{self.n_materials}"
                     f" {self.n_layers} {self.basic_info['CosAlfa']}\n")

        # Write block B: WATER FLOW INFORMATION
        lines.append(string.format("B: WATER FLOW INFORMATION ", "*", "<", 72))
        lines.append("MaxIt  TolTh  TolH   (maximum number of iterations and "
                     "tolerances)\n")
        variables = ["MaxIt", "TolTh", "TolH"]
        lines.append(
            "   ".join([str(self.water_flow[var]) for var in variables]))
        lines.append("\n")

        vars_list = [["TopInf", "WLayer", "KodTop", "lInitW", "\n"],
                     ["BotInf", "qGWLF", "FreeD", "SeepF", "KodBot", "qDrain",
                      "hSeep", "\n"]]

        upper_condition = (self.water_flow["KodTop"] < 0
                           and not self.water_flow["TopInf"])

        lower_condition = ((self.water_flow["KodBot"] < 0)
                           and not self.water_flow["BotInf"]
                           and not self.water_flow["qGWLF"]
                           and not self.water_flow["FreeD"]
                           and not self.water_flow["SeepF"])

        if upper_condition or lower_condition:
            vars_list.append(["rTop", "rBot", "rRoot", "\n"])

        if self.water_flow["qGWLF"]:
            vars_list.append(["GWL0L", "Aqh", "Bqh", "\n"])

        vars_list.append(["ha", "hb", "\n"])
        vars_list.append(["iModel", "iHyst", "\n"])

        if self.water_flow["iHyst"] > 0:
            vars_list.append(["iKappa", "\n"])

        for variables in vars_list:
            lines.append("  ".join(variables))
            values = []
            for var in variables[:-1]:
                val = self.water_flow[var]
                if val is True:
                    values.append("t")
                elif val is False:
                    values.append("f")
                else:
                    values.append(f"{val}")
            values.append("\n")
            lines.append(" ".join(values))

        if self.drains:
            self.logger.error("Drains are currently not Implemented.")
            raise

        # Write the material parameters
        lines.append(self.materials["water"].to_string(index=False))
        lines.append("\n")

        # Write BLOCK C: TIME INFORMATION
        lines.append(string.format("C: TIME INFORMATION ", "*", "<", 72))
        vars_list = [
            ["dt", "dtMin", "dtMax", "dMul", "dMul2", "ItMin", "ItMax",
             "MPL", "\n"], ["tInit", "tMax", "\n"],
            ["lPrint", "nPrintSteps", "tPrintInterval", "lEnter", "\n"]]
        for variables in vars_list:
            lines.append(" ".join(variables))
            values = []
            for var in variables[:-1]:
                val = self.time_info[var]
                if val is True:
                    values.append("t")
                elif val is False:
                    values.append("f")
                else:
                    values.append(str(val))
            values.append("\n")
            lines.append(" ".join(values))

        lines.append("TPrint(1),TPrint(2),...,TPrint(MPL)\n")
        for i in range(int(len(self.times) / 6) + 1):
            lines.append(
                " ".join([str(time) for time in self.times[i * 6:i * 6 + 6]]))
            lines.append("\n")

                # Write BLOCK D: Root Growth Information
        if self.basic_info["lRoot"]:
            lines.append(
                string.format("D: ROOT GROWTH INFORMATION ", "*", "<", 72))
            lines.append(f"iRootDepthEntry\n{self.root_growth['iRootIn']}\n")
            d = self.root_growth.copy()
            d.pop("iRootIn")
            d["\n"] = "\n"
            lines.append("    ".join(d.keys()))
            lines.append("    ".join(f"{p}" for p in d.values()))

        # Write Block E - Heat transport information
        if self.basic_info["lTemp"]:
            lines.append(string.format("E: HEAT TRANSPORT INFORMATION ",
                                       "*", "<", 72))
            lines.append(self.heat_parameters.to_string(index=False)).pop("nGrowth")
            
            for i in list(range(len(days))):
                lines.append(str(days[i]))
                lines.append(" ")
                lines.append(str(depths[i]))
                lines.append("\n")

            lines.append(
                "\n tAmpl tPeriod Campbell SnowMF lDummy lDummy lDummy "
                "lDummy lDummy\n"
                "{} {} {} {} f f f f f\n"
                "kTopT TTop kBotT TBot\n"
                "{} {} {} {}\n".format(self.heat_transport["Ampl"],
                                       self.heat_transport["tPeriod"],
                                       self.heat_transport["iCampbell"],
                                       self.heat_transport["SnowMF"],
                                       self.heat_transport["kTopT"],
                                       self.heat_transport["tTop"],
                                       self.heat_transport["kBotT"],
                                       self.heat_transport["tBot"]))

        # Write Block F - Solute transport information
        if self.basic_info["lChem"]:
            lines.append(string.format("F: SOLUTE TRANSPORT INFORMATION ",
                                       "*", "<", 72))
            lines.append(" Epsi lUpW lArtD lTDep cTolA cTolR MaxItC PeCr "
                         "No.Solutes lTort iBacter lFiltr nChPar\n"
                         "{} {} {} {} {} {} {} {} {} {} {} {} {}\n"
                         "iNonEqul lWatDep lDualNEq lInitM lInitEq lTort "
                         "lDummy lDummy lDummy lDummy lCFTr\n"
                         "{} {} {} {} {} {} f f f f f\n".format(
                self.solute_transport["Epsi"],
                "t" if self.solute_transport["lUpW"] else "f",
                "t" if self.solute_transport["lArtD"] else "f",
                "t" if self.solute_transport["ltDep"] else "f",
                self.solute_transport["cTolA"],
                self.solute_transport["cTolR"],
                self.solute_transport["MaxItC"],
                self.solute_transport["PeCr"],
                self.n_solutes,
                "t" if self.solute_transport["lTort"] else "f",
                self.solute_transport["iBacter"],
                "t" if self.solute_transport["lFiltr"] else "f",
                self.get_empty_solute_df().columns.size + 2,
                self.solute_transport["iNonEqual"],
                "t" if self.solute_transport["lWatDep"] else "f",
                "t" if self.solute_transport["lDualEq"] else "f",
                "f", "f",
                "t" if self.solute_transport["lTort"] else "f"
            ))

            # Write the material parameters
            lines.append(self.materials["solute"].to_string(index=False))
            lines.append("\n")

            for sol in self.solutes:
                lines.append(f"DifW DifG\n{sol['difw']} {sol['difg']}\n"
                             f"{sol['data'].to_string(index=False)}\n")

            lines.append("kTopSolute SolTop kBotSolute SolBot\n"
                         "{} {} {} {}\n".format(
                self.solute_transport["kTopCh"],
                " ".join([f"{s['top_conc']}" for s in self.solutes]),
                self.solute_transport["kBotCh"],
                " ".join([f"{s['bot_conc']}" for s in self.solutes])))
            if self.solute_transport["kTopCh"] == -2:
                lines.append("dSurf cAtm\n""{} {}\n".format(
                    self.solute_transport["dSurf"],
                    self.solute_transport["cAtm"]))

            lines.append("tPulse\n{}\n".format(
                self.solute_transport["tPulse"]))

        # Write Block G - Root water uptake information
        if self.basic_info["lSink"]:
            lines.append(string.format("G: ROOT WATER UPTAKE INFORMATION ",
                                       "*", "<", 72))
            vars_list = [["iMoSink", "cRootMax", "OmegaC", "\n"]]

            if self.root_uptake["iMoSink"] == 0:
                vars_list.append(
                    ["P0", "P2H", "P2L", "P3", "r2H", "r2L", "\n"])
            elif self.root_uptake["iMoSink"] == 1:
                vars_list.append(["P50", "P3", "\n"])

            for variables in vars_list:
                lines.append(" ".join(variables))
                lines.append("    ".join(f"{self.root_uptake[var]}" for var in
                                         variables[:-1]))
                lines.append("\n")

            lines.append("POptm(1),POptm(2),...,POptm(NMat)\n")
            lines.append("    ".join(f"{p}" for p in self.root_uptake[
                "POptm"]))
            lines.append("\n")

            if self.basic_info["lChem"]:
                lines.append("Solute Reduction\nf\n")

        # Write Block J - Inverse solution information
        if self.basic_info["lInverse"]:
            raise NotImplementedError("The inverse modeling module from "
                                      "Hydrus-1D will not be supported. "
                                      "Python packages are used for this.")

        # Write Block K – Carbon dioxide transport information

        # Write Block M – Meteorological information
        if self.basic_info["lMeteo"]:
            raise NotImplementedError

        # Write END statement
        lines.append(string.format("END OF INPUT FILE SELECTOR.IN ",
                                   "*", "<", 72))

        # Write the actual file
        fname = os.path.join(self.ws_name, fname)
        with open(fname, "w") as file:
            file.writelines(lines)

        self.logger.info("Successfully wrote %s", fname)



[ 0 ,   1 ,   2 ,   3 ,   4 ,   5 ,   6 ,   7 ,   8 ,   9 ,   1 0 ,   1 1 ,   1 2 ,   1 3 ,   1 4 ,   1 5 ,   1 6 ,   1 7 ,   1 8 ,   1 9 ]


In [ ]:
#Editing function

def write_selector_modified(self, fname="SELECTOR.IN"):
        """
        Write the SELECTOR.IN file.

        Parameters
        ----------
        fname: str, optional
            String with the filename. Written to the workspace folder ('ws').

        """
        self._set_bc_settings()

        # Create Header string
        string = "*** BLOCK {:{}{}{}}\n"

        # Write block A: BASIC INFORMATION
        lines = [
            f"Pcp_File_Version={self.basic_info['iVer']}\n"
            f"{string.format('A: BASIC INFORMATION ', '*', '<', 72)}"
            f"{self.basic_info['Hed']}\n{self.description}\n"
            f"LUnit TUnit MUnit\n{self.basic_info['LUnit']}\n"
            f"{self.basic_info['TUnit']}\n{self.basic_info['MUnit']}\n"
        ]

        vars_list = [["lWat", "lChem", "lTemp", "lSink", "lRoot", "lShort",
                      "lWDep", "lScreen", "AtmInf", "lEquil", "lInverse",
                      "\n"],
                     ["lSnow", "lHP1", "lMeteo", "lVapor", "lActRSU", "lFlux",
                      "lIrrig", "\n"]]

        for variables in vars_list:
            lines.append("  ".join(variables))
            lines.append("  ".join("t" if self.basic_info[var] else "f" for
                                   var in variables[:-1]))
            lines.append("\n")

        lines.append(f"NMat NLay CosAlfa \n{self.n_materials}"
                     f" {self.n_layers} {self.basic_info['CosAlfa']}\n")

        # Write block B: WATER FLOW INFORMATION
        lines.append(string.format("B: WATER FLOW INFORMATION ", "*", "<", 72))
        lines.append("MaxIt  TolTh  TolH   (maximum number of iterations and "
                     "tolerances)\n")
        variables = ["MaxIt", "TolTh", "TolH"]
        lines.append(
            "   ".join([str(self.water_flow[var]) for var in variables]))
        lines.append("\n")

        vars_list = [["TopInf", "WLayer", "KodTop", "lInitW", "\n"],
                     ["BotInf", "qGWLF", "FreeD", "SeepF", "KodBot", "qDrain",
                      "hSeep", "\n"]]

        upper_condition = (self.water_flow["KodTop"] < 0
                           and not self.water_flow["TopInf"])

        lower_condition = ((self.water_flow["KodBot"] < 0)
                           and not self.water_flow["BotInf"]
                           and not self.water_flow["qGWLF"]
                           and not self.water_flow["FreeD"]
                           and not self.water_flow["SeepF"])

        if upper_condition or lower_condition:
            vars_list.append(["rTop", "rBot", "rRoot", "\n"])

        if self.water_flow["qGWLF"]:
            vars_list.append(["GWL0L", "Aqh", "Bqh", "\n"])

        vars_list.append(["ha", "hb", "\n"])
        vars_list.append(["iModel", "iHyst", "\n"])

        if self.water_flow["iHyst"] > 0:
            vars_list.append(["iKappa", "\n"])

        for variables in vars_list:
            lines.append("  ".join(variables))
            values = []
            for var in variables[:-1]:
                val = self.water_flow[var]
                if val is True:
                    values.append("t")
                elif val is False:
                    values.append("f")
                else:
                    values.append(f"{val}")
            values.append("\n")
            lines.append(" ".join(values))

        if self.drains:
            self.logger.error("Drains are currently not Implemented.")
            raise

        # Write the material parameters
        lines.append(self.materials["water"].to_string(index=False))
        lines.append("\n")

        # Write BLOCK C: TIME INFORMATION
        lines.append(string.format("C: TIME INFORMATION ", "*", "<", 72))
        vars_list = [
            ["dt", "dtMin", "dtMax", "dMul", "dMul2", "ItMin", "ItMax",
             "MPL", "\n"], ["tInit", "tMax", "\n"],
            ["lPrint", "nPrintSteps", "tPrintInterval", "lEnter", "\n"]]
        for variables in vars_list:
            lines.append(" ".join(variables))
            values = []
            for var in variables[:-1]:
                val = self.time_info[var]
                if val is True:
                    values.append("t")
                elif val is False:
                    values.append("f")
                else:
                    values.append(str(val))
            values.append("\n")
            lines.append(" ".join(values))

        lines.append("TPrint(1),TPrint(2),...,TPrint(MPL)\n")
        for i in range(int(len(self.times) / 6) + 1):
            lines.append(
                " ".join([str(time) for time in self.times[i * 6:i * 6 + 6]]))
            lines.append("\n")

        # Write BLOCK D: Root Growth Information
        if self.basic_info["lRoot"]:
            lines.append(
                string.format("D: ROOT GROWTH INFORMATION ", "*", "<", 72))
            lines.append(f"iRootDepthEntry\n{self.root_growth['iRootIn']}\n")
            
            lines.append("nGrowth\n")
            lines.append(f"{self.root_growth['nGrowth']}\n")

            width = len(str(self.root_growth['nGrowth']))

            lines.append(str("Time RootDepth\n"))

            for i in list(range(len(days))):
                lines.append(str(self.root_growth['tGrowth'][i]+1).ljust(width))
                lines.append(" ")
                lines.append(str(self.root_growth['RootDepth'][i]))
                lines.append("\n")

        # Write Block E - Heat transport information
        if self.basic_info["lTemp"]:
            lines.append(string.format("E: HEAT TRANSPORT INFORMATION ",
                                       "*", "<", 72))
            lines.append(self.heat_parameters.to_string(index=False)).pop("nGrowth")
            
            for i in list(range(len(days))):
                lines.append(str(days[i]))
                lines.append(" ")
                lines.append(str(depths[i]))
                lines.append("\n")

            lines.append(
                "\n tAmpl tPeriod Campbell SnowMF lDummy lDummy lDummy "
                "lDummy lDummy\n"
                "{} {} {} {} f f f f f\n"
                "kTopT TTop kBotT TBot\n"
                "{} {} {} {}\n".format(self.heat_transport["Ampl"],
                                       self.heat_transport["tPeriod"],
                                       self.heat_transport["iCampbell"],
                                       self.heat_transport["SnowMF"],
                                       self.heat_transport["kTopT"],
                                       self.heat_transport["tTop"],
                                       self.heat_transport["kBotT"],
                                       self.heat_transport["tBot"]))

        # Write Block F - Solute transport information
        if self.basic_info["lChem"]:
            lines.append(string.format("F: SOLUTE TRANSPORT INFORMATION ",
                                       "*", "<", 72))
            lines.append(" Epsi lUpW lArtD lTDep cTolA cTolR MaxItC PeCr "
                         "No.Solutes lTort iBacter lFiltr nChPar\n"
                         "{} {} {} {} {} {} {} {} {} {} {} {} {}\n"
                         "iNonEqul lWatDep lDualNEq lInitM lInitEq lTort "
                         "lDummy lDummy lDummy lDummy lCFTr\n"
                         "{} {} {} {} {} {} f f f f f\n".format(
                self.solute_transport["Epsi"],
                "t" if self.solute_transport["lUpW"] else "f",
                "t" if self.solute_transport["lArtD"] else "f",
                "t" if self.solute_transport["ltDep"] else "f",
                self.solute_transport["cTolA"],
                self.solute_transport["cTolR"],
                self.solute_transport["MaxItC"],
                self.solute_transport["PeCr"],
                self.n_solutes,
                "t" if self.solute_transport["lTort"] else "f",
                self.solute_transport["iBacter"],
                "t" if self.solute_transport["lFiltr"] else "f",
                self.get_empty_solute_df().columns.size + 2,
                self.solute_transport["iNonEqual"],
                "t" if self.solute_transport["lWatDep"] else "f",
                "t" if self.solute_transport["lDualEq"] else "f",
                "f", "f",
                "t" if self.solute_transport["lTort"] else "f"
            ))

            # Write the material parameters
            lines.append(self.materials["solute"].to_string(index=False))
            lines.append("\n")

            for sol in self.solutes:
                lines.append(f"DifW DifG\n{sol['difw']} {sol['difg']}\n"
                             f"{sol['data'].to_string(index=False)}\n")

            lines.append("kTopSolute SolTop kBotSolute SolBot\n"
                         "{} {} {} {}\n".format(
                self.solute_transport["kTopCh"],
                " ".join([f"{s['top_conc']}" for s in self.solutes]),
                self.solute_transport["kBotCh"],
                " ".join([f"{s['bot_conc']}" for s in self.solutes])))
            if self.solute_transport["kTopCh"] == -2:
                lines.append("dSurf cAtm\n""{} {}\n".format(
                    self.solute_transport["dSurf"],
                    self.solute_transport["cAtm"]))

            lines.append("tPulse\n{}\n".format(
                self.solute_transport["tPulse"]))

        # Write Block G - Root water uptake information
        if self.basic_info["lSink"]:
            lines.append(string.format("G: ROOT WATER UPTAKE INFORMATION ",
                                       "*", "<", 72))
            vars_list = [["iMoSink", "cRootMax", "OmegaC", "\n"]]

            if self.root_uptake["iMoSink"] == 0:
                vars_list.append(
                    ["P0", "P2H", "P2L", "P3", "r2H", "r2L", "\n"])
            elif self.root_uptake["iMoSink"] == 1:
                vars_list.append(["P50", "P3", "\n"])

            for variables in vars_list:
                lines.append(" ".join(variables))
                lines.append("    ".join(f"{self.root_uptake[var]}" for var in
                                         variables[:-1]))
                lines.append("\n")

            lines.append("POptm(1),POptm(2),...,POptm(NMat)\n")
            lines.append("    ".join(f"{p}" for p in self.root_uptake[
                "POptm"]))
            lines.append("\n")

            if self.basic_info["lChem"]:
                lines.append("Solute Reduction\nf\n")

        # Write Block J - Inverse solution information
        if self.basic_info["lInverse"]:
            raise NotImplementedError("The inverse modeling module from "
                                      "Hydrus-1D will not be supported. "
                                      "Python packages are used for this.")

        # Write Block K – Carbon dioxide transport information

        # Write Block M – Meteorological information
        if self.basic_info["lMeteo"]:
            raise NotImplementedError

        # Write END statement
        lines.append(string.format("END OF INPUT FILE SELECTOR.IN ",
                                   "*", "<", 72))

        # Write the actual file
        fname = os.path.join(self.ws_name, fname)
        with open(fname, "w") as file:
            file.writelines(lines)

        self.logger.info("Successfully wrote %s", fname)



In [ ]:
def add_root_growth(self, irootin=0, ngrowth=None, tgrowth=None,
                        rootdepth=None, irfak=None, trmin=None, trmed=None,
                        trmax=None, xrmin=None, xrmed=None, xrmax=None,
                        trperiod=None):
        """
        Method to add root growth to the model.

        Parameters
        ----------
        irootin: int
            0 = (default) The root depth is specified together with other
            time-variable boundary condition, such as atmospheric fluxes.
            1 = the root depth is given in a table
            2 = the root depth is calculated using the growth function.
        ngrowth: int
            Number of data points in the root depth table. Only used when
            irootin = 1.
        tgrowth: float, optional
            Days. Only used when irootin = 1.
        rootdepth: list of float, optional
            Rooting depth [L]. List has a length of ngrowth. Only used when
            irootin = 1.
        irfak: int, optional
            Method to calculate the root growth factor, r. Only used when
            irootin = 2.
            0= the root growth factor is calculated from given data [xRMed,
            tRMed].
            1 = the root growth factor is calculated based on the assumption
            that 50% of the rooting depth, (xRMax+xRMin)/2., is reached at
            the midpoint of the growing season, (tRMin+tRHarv)/2.
        trmin: float, optional
            Initial time of the root growth period [T]. Only used when
            irootin = 2.
        trmed: float, optional
            Time of known rooting depth (set equal to zero if iRFak=1) [T].
            Only used when irootin = 2.
        trmax: float, optional
            Time at the end of the root water uptake period [T]. Only used when
            irootin = 2.
        xrmin: float, optional
            Initial value of the rooting depth at the beginning of the
            growth period (recommended value = 1 cm) [L]. Only used when
            irootin = 2.
        xrmed: float, optional
            Value of known rooting depth (set equal to zero if iRFak=1) [L].
            Only used when irootin = 2.
        xrmax: float, optional
            Maximum rooting depth, which may be reached at infinite time [L].
            Only used when irootin = 2.
        trperiod: float, optional
            Time period at which the growth function repeats itself. Only
            used when irootin = 2.

        """

        # Store the root growth information depending on the model.
        if irootin == 0:
            root_growth = {
                "iRootIn": irootin
            }
        elif irootin == 1:
            root_growth = {
                "iRootIn": irootin,
                "nGrowth": ngrowth,
                "tGrowth": tgrowth,
                "RootDepth": rootdepth
            }
        elif irootin == 2:
            root_growth = {
                "iRootIn": irootin,
                "iRFak": irfak,
                "tRMin": trmin,
                "tRMed": trmed,
                "tRMax": trmax,
                "xRMin": xrmin,
                "xRMed": xrmed,
                "xRMax": xrmax,
                "tRPeriod": trperiod
            }
            if irfak == 1:
                root_growth["tRMed"] = 0
                root_growth["xRMed"] = 0
        else:
            raise Warning("Option %s for irootin is not support in Hydrus."
                          % irootin)

        if self.root_growth is None:
            self.root_growth = root_growth
            self.basic_info["lRoot"] = True
        else:
            raise Warning("Root growth model already exists. Please delete "
                          "the old root growth model first using "
                          "ml.del_root_growth().")
